# INSERT, SELECT a agregační funkce

V předchozích noteboocích jsme se naučili vytvářet (`CREATE`), upravovat (`ALTER`) a mazat (`DROP`) tabulky. Nyní se zaměříme na práci se **samotnými daty** — naučíme se data do tabulky **vkládat** (`INSERT`), **číst** (`SELECT`) a provádět nad nimi jednoduché **výpočty** pomocí agregačních funkcí.

---

## Přehled příkazů v tomto notebooku

| Příkaz / Funkce | Kategorie | Popis |
|---|---|---|
| `INSERT INTO` | DML | Vloží nový záznam (řádek) do tabulky |
| `SELECT` | DQL | Přečte (vybere) data z tabulky |
| `AS` | DQL | Přejmenuje sloupec ve výstupu (alias) |
| `COUNT()` | Agregační | Počet záznamů |
| `SUM()` | Agregační | Součet hodnot |
| `AVG()` | Agregační | Aritmetický průměr |
| `MIN()` | Agregační | Minimální hodnota |
| `MAX()` | Agregační | Maximální hodnota |

## Instalace balíčku

In [ ]:
# Instalace knihovny (stačí spustit jednou)
! pip install mysql-connector-python

# Připojení k databázi

Než začneme s databází pracovat, musíme se k ní připojit. Přihlašovací údaje importujeme z modulu `pripojeni.py` (nikdy je nepíšeme přímo do kódu).

In [ ]:
import mysql.connector
from pripojeni import *  # importuje konstanty HOST, USER, PASSWORD, DATABASE

# Připojení k databázi
mydb = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DATABASE
)

# Kurzor — objekt, přes který posíláme SQL příkazy a čteme výsledky
mycursor = mydb.cursor()

## Příprava ukázkové tabulky

Pro následující příklady si vytvoříme tabulku `automobily` se stejnou strukturou jako v předchozích noteboocích:

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS automobily")

mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL UNIQUE,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(50) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")

mydb.commit()
print("✅ Tabulka 'automobily' vytvořena.")

---

# INSERT INTO — vkládání dat

Příkaz `INSERT INTO` slouží k **vložení nového záznamu** (řádku) do tabulky.

## Syntaxe

```sql
INSERT INTO nazev_tabulky (sloupec_1, sloupec_2, sloupec_3) VALUES
    (hodnota_1, hodnota_2, hodnota_3);
```

Jedním příkazem lze vložit i **více záznamů najednou** — hodnoty oddělíme čárkou:

```sql
INSERT INTO nazev_tabulky (sloupec_1, sloupec_2, sloupec_3) VALUES
    (hodnota_1, hodnota_2, hodnota_3),
    (hodnota_1, hodnota_2, hodnota_3),
    (hodnota_1, hodnota_2, hodnota_3);
```

### Pravidla

- **Pořadí hodnot** musí odpovídat pořadí uvedených sloupců.
- **Datové typy** musí odpovídat definici sloupce — text musí být v uvozovkách (`'text'`), čísla bez uvozovek.
- **Datum** se zapisuje ve formátu `'YYYY-MM-DD'` (ISO 8601).
- Sloupce s `AUTO_INCREMENT` nemusíme uvádět — databáze doplní hodnotu automaticky.
- Sloupce s `DEFAULT` hodnotou nemusíme uvádět — použije se výchozí hodnota.
- Po `INSERT` je nutné zavolat `mydb.commit()` — jinak se data neuloží!

> **⚠️ Pozor:** V Pythonu je celý SQL dotaz uvnitř trojitých uvozovek (`"""..."""`). Textové hodnoty v SQL proto musíme psát v **jednoduchých** uvozovkách (`'text'`), aby nedošlo ke konfliktu.

### Příklad

In [ ]:
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 4, 190, 3, 'B', '2000-09-09'),
        ('2B22222', 2, 220, 2, 'B', '2020-01-01'),
        ('3C33333', 5, 160, 5, 'B', '2015-06-15'),
        ('4D44444', 3, 250, 2, 'A', '2022-03-20'),
        ('5E55555', 8, 120, 10, 'D', '2018-11-30')
""")

# commit() potvrdí provedení příkazu — bez něj se data neuloží!
mydb.commit()

print(f"✅ Vloženo {mycursor.rowcount} záznamů.")

> **Tip:** Vlastnost `mycursor.rowcount` vrací počet řádků, které byly posledním příkazem ovlivněny.

> **Tip:** Sloupec `id` jsme neuváděli — díky `AUTO_INCREMENT` se hodnoty 1, 2, 3, … doplní automaticky. Stejně tak `nutna_kvalifikace` bychom mohli vynechat u aut s kvalifikací `'B'`, protože má nastavený `DEFAULT 'B'`.

---

# SELECT — čtení dat

Příkaz `SELECT` slouží ke **čtení (výběru) dat** z tabulky. Je to nejpoužívanější SQL příkaz.

## Syntaxe

```sql
/* Výběr všech sloupců */
SELECT * FROM nazev_tabulky;

/* Výběr konkrétních sloupců */
SELECT sloupec_1, sloupec_2 FROM nazev_tabulky;
```

| Zápis | Význam |
|---|---|
| `SELECT *` | Vybere **všechny sloupce** |
| `SELECT sloupec_1, sloupec_2` | Vybere **pouze uvedené sloupce** |

> **Důležité:** Na rozdíl od `INSERT` nemusíme po `SELECT` volat `commit()`, protože do tabulky nic nezapisujeme — pouze čteme.

### Zpracování výsledků v Pythonu

Po provedení `SELECT` musíme výsledky **načíst z kurzoru** do proměnné:

| Metoda | Popis |
|---|---|
| `mycursor.fetchall()` | Vrátí **všechny** řádky jako seznam tuplů |
| `mycursor.fetchone()` | Vrátí **jeden** řádek (další volání vrátí další) |
| `mycursor.fetchmany(n)` | Vrátí **n** řádků |

### Příklad — výběr všech sloupců

In [ ]:
mycursor.execute("""SELECT * FROM automobily""")

myresult = mycursor.fetchall()  # načte všechny řádky do seznamu

# Jednoduchý výpis — seznam tuplů
for radek in myresult:
    print(radek)

### Příklad — výběr konkrétních sloupců s přehledným výpisem

In [ ]:
mycursor.execute("""SELECT id, spz, max_rychlost, datum_vyroby FROM automobily""")

myresult = mycursor.fetchall()

# Přehledný výpis pomocí rozbalení (unpacking) tuplů
for id, spz, max_rychlost, datum_vyroby in myresult:
    print(f"Auto #{id}: SPZ={spz}, max. rychlost={max_rychlost} km/h, vyrobeno={datum_vyroby}")

> **Tip:** Počet proměnných ve `for` cyklu musí odpovídat počtu sloupců v `SELECT`. Pokud se počet neshoduje, Python vyhodí `ValueError`.

## Alias — přejmenování sloupce (`AS`)

Pomocí klíčového slova `AS` můžeme ve výstupu **přejmenovat sloupec** (vytvořit alias). Data v tabulce se nezmění — alias platí pouze pro daný dotaz.

```sql
SELECT sloupec AS "Nový název" FROM nazev_tabulky;
```

### Příklad

In [ ]:
mycursor.execute("""
    SELECT
        spz AS "SPZ vozidla",
        max_rychlost AS "Maximalni rychlost [km/h]",
        pocet_sedadel AS "Pocet sedadel"
    FROM automobily
""")

# Názvy sloupců (aliasy) lze získat z kurzoru
nazvy_sloupcu = [popis[0] for popis in mycursor.description]
print(nazvy_sloupcu)

myresult = mycursor.fetchall()
for radek in myresult:
    print(radek)

> **Tip:** Aliasy jsou užitečné zejména při agregačních funkcích — bez aliasu se sloupec pojmenuje např. `AVG(max_rychlost)`, což není přehledné.

---

# Agregační funkce

Agregační funkce provádějí **výpočet nad množinou hodnot** a vrací **jeden výsledek**. Používají se spolu s příkazem `SELECT`.

## Přehled agregačních funkcí

| Funkce | Popis | Příklad |
|---|---|---|
| `COUNT(sloupec)` | Počet **neprázdných** (`NOT NULL`) hodnot ve sloupci | `COUNT(*)` — počet všech řádků |
| `SUM(sloupec)` | **Součet** všech hodnot (pouze číselné sloupce) | `SUM(nosnost)` |
| `AVG(sloupec)` | **Aritmetický průměr** hodnot (pouze číselné sloupce) | `AVG(max_rychlost)` |
| `MIN(sloupec)` | **Nejmenší** hodnota ve sloupci | `MIN(datum_vyroby)` |
| `MAX(sloupec)` | **Největší** hodnota ve sloupci | `MAX(max_rychlost)` |

## Syntaxe

```sql
SELECT FUNKCE(sloupec) FROM nazev_tabulky;
```

Lze kombinovat více agregačních funkcí v jednom dotazu:

```sql
SELECT
    COUNT(*) AS "Počet aut",
    AVG(max_rychlost) AS "Průměrná rychlost",
    MIN(max_rychlost) AS "Nejpomalejší",
    MAX(max_rychlost) AS "Nejrychlejší"
FROM automobily;
```

> **Pozn.:** `COUNT(*)` počítá **všechny řádky** (včetně řádků s `NULL` hodnotami). `COUNT(sloupec)` počítá pouze řádky, kde daný sloupec **není `NULL`**.

### Příklad — jednotlivé agregační funkce

In [ ]:
# Počet záznamů
mycursor.execute("""SELECT COUNT(*) AS 'pocet' FROM automobily""")
vysledek = mycursor.fetchone()
print(f"Počet automobilů: {vysledek[0]}")

# Průměrná maximální rychlost
mycursor.execute("""SELECT AVG(max_rychlost) AS 'prumer' FROM automobily""")
vysledek = mycursor.fetchone()
print(f"Průměrná max. rychlost: {vysledek[0]} km/h")

# Součet nosností
mycursor.execute("""SELECT SUM(nosnost) AS 'soucet' FROM automobily""")
vysledek = mycursor.fetchone()
print(f"Celková nosnost: {vysledek[0]} t")

# Minimum a maximum
mycursor.execute("""SELECT MIN(max_rychlost) AS 'min', MAX(max_rychlost) AS 'max' FROM automobily""")
vysledek = mycursor.fetchone()
print(f"Nejpomalejší: {vysledek[0]} km/h, Nejrychlejší: {vysledek[1]} km/h")

### Příklad — kombinace více agregačních funkcí

In [ ]:
mycursor.execute("""
    SELECT
        COUNT(*) AS 'Pocet aut',
        AVG(max_rychlost) AS 'Prumerna rychlost',
        SUM(nosnost) AS 'Celkova nosnost',
        MIN(datum_vyroby) AS 'Nejstarsi',
        MAX(datum_vyroby) AS 'Nejnovejsi'
    FROM automobily
""")

pocet, prumer, nosnost, nejstarsi, nejnovejsi = mycursor.fetchone()

print(f"Počet aut:           {pocet}")
print(f"Průměrná rychlost:   {prumer} km/h")
print(f"Celková nosnost:     {nosnost} t")
print(f"Nejstarší auto:      {nejstarsi}")
print(f"Nejnovější auto:     {nejnovejsi}")

## Odpojení a úklid

Na konci ukázkové části smažeme tabulku a odpojíme se:

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS automobily")
mydb.commit()

mycursor.close()
mydb.close()

print("✅ Odpojení proběhlo úspěšně.")

---

# Cvičení

Zde bude následovat série úkolů, díky kterým si ověříte, zda jste látku pochopili.

> V každém cvičení se nejprve připojte k databázi pomocí konstant z modulu `pripojeni`. Na konci se vždy odpojte.

## Cvičení 1 — INSERT

Tabulka `mesta` je pro vás již vytvořena (viz kód níže). Doplňte příkaz `INSERT INTO`, kterým do ní vložíte **alespoň 5 měst**.

Tabulka `mesta` má sloupce:
- `id` — INT, PRIMARY KEY, AUTO_INCREMENT
- `nazev` — CHAR(30), NOT NULL
- `okres` — CHAR(30), NOT NULL
- `kraj` — CHAR(30), NOT NULL
- `pocet_obyvatel` — INT, NOT NULL

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
✅ Data vložena správně — v tabulce je X záznamů. (kde X >= 5)
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! ---
mycursor.execute("DROP TABLE IF EXISTS mesta")
mycursor.execute("""
    CREATE TABLE mesta (
        id INT AUTO_INCREMENT PRIMARY KEY,
        nazev CHAR(30) NOT NULL,
        okres CHAR(30) NOT NULL,
        kraj CHAR(30) NOT NULL,
        pocet_obyvatel INT NOT NULL
    )
""")
mydb.commit()

# TODO: Vložte alespoň 5 měst pomocí INSERT INTO →


mydb.commit()

# --- tuto část neměnit! ---
mycursor.execute("SELECT COUNT(*) FROM mesta")
pocet = mycursor.fetchone()[0]
if pocet >= 5:
    print(f"✅ Data vložena správně — v tabulce je {pocet} záznamů.")
else:
    print(f"❌ V tabulce je pouze {pocet} záznamů (minimum je 5).")

mycursor.execute("DROP TABLE mesta")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 2 — SELECT

Z tabulky `mesta` (vytvořené a naplněné v kódu níže) vypište **všechny záznamy**. Použijte přehledný výpis pomocí `for` cyklu s rozbalením tuplů.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
1: Praha (okres: Praha, kraj: Hlavni mesto Praha, obyvatel: 1309000)
2: Brno (okres: Brno-mesto, kraj: Jihomoravsky, obyvatel: 382000)
3: Ostrava (okres: Ostrava-mesto, kraj: Moravskoslezsky, obyvatel: 285000)
4: Plzen (okres: Plzen-mesto, kraj: Plzensky, obyvatel: 175000)
5: Kladno (okres: Kladno, kraj: Stredocesky, obyvatel: 69000)
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS mesta")
mycursor.execute("""
    CREATE TABLE mesta (
        id INT AUTO_INCREMENT PRIMARY KEY,
        nazev CHAR(30) NOT NULL,
        okres CHAR(30) NOT NULL,
        kraj CHAR(30) NOT NULL,
        pocet_obyvatel INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO mesta (nazev, okres, kraj, pocet_obyvatel) VALUES
        ('Praha', 'Praha', 'Hlavni mesto Praha', 1309000),
        ('Brno', 'Brno-mesto', 'Jihomoravsky', 382000),
        ('Ostrava', 'Ostrava-mesto', 'Moravskoslezsky', 285000),
        ('Plzen', 'Plzen-mesto', 'Plzensky', 175000),
        ('Kladno', 'Kladno', 'Stredocesky', 69000)
""")
mydb.commit()

# TODO: Vypište všechny záznamy z tabulky mesta →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE mesta")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 3 — SELECT s aliasy

Z tabulky `mesta` vypište **pouze názvy měst a počty obyvatel**. Ve výpisu použijte aliasy viz. očekávaný výstup.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Město: Praha, Obyvatel: 1309000
Město: Brno, Obyvatel: 382000
Město: Ostrava, Obyvatel: 285000
Město: Plzen, Obyvatel: 175000
Město: Kladno, Obyvatel: 69000
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS mesta")
mycursor.execute("""
    CREATE TABLE mesta (
        id INT AUTO_INCREMENT PRIMARY KEY,
        nazev CHAR(30) NOT NULL,
        okres CHAR(30) NOT NULL,
        kraj CHAR(30) NOT NULL,
        pocet_obyvatel INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO mesta (nazev, okres, kraj, pocet_obyvatel) VALUES
        ('Praha', 'Praha', 'Hlavni mesto Praha', 1309000),
        ('Brno', 'Brno-mesto', 'Jihomoravsky', 382000),
        ('Ostrava', 'Ostrava-mesto', 'Moravskoslezsky', 285000),
        ('Plzen', 'Plzen-mesto', 'Plzensky', 175000),
        ('Kladno', 'Kladno', 'Stredocesky', 69000)
""")
mydb.commit()

# TODO: Vypište pouze názvy měst a počty obyvatel s aliasy →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE mesta")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 4 — Agregační funkce

Z tabulky `mesta` zjistěte a vypište:
1. **Počet** měst v tabulce
2. **Průměrný** počet obyvatel
3. **Nejmenší** počet obyvatel
4. **Největší** počet obyvatel
5. **Celkový součet** obyvatel všech měst

Použijte **jeden SQL dotaz** se všemi agregačními funkcemi najednou.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Počet měst:          5
Průměr obyvatel:     444000.0
Minimum obyvatel:    69000
Maximum obyvatel:    1309000
Celkem obyvatel:     2220000
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS mesta")
mycursor.execute("""
    CREATE TABLE mesta (
        id INT AUTO_INCREMENT PRIMARY KEY,
        nazev CHAR(30) NOT NULL,
        okres CHAR(30) NOT NULL,
        kraj CHAR(30) NOT NULL,
        pocet_obyvatel INT NOT NULL
    )
""")
mycursor.execute("""
    INSERT INTO mesta (nazev, okres, kraj, pocet_obyvatel) VALUES
        ('Praha', 'Praha', 'Hlavni mesto Praha', 1309000),
        ('Brno', 'Brno-mesto', 'Jihomoravsky', 382000),
        ('Ostrava', 'Ostrava-mesto', 'Moravskoslezsky', 285000),
        ('Plzen', 'Plzen-mesto', 'Plzensky', 175000),
        ('Kladno', 'Kladno', 'Stredocesky', 69000)
""")
mydb.commit()

# TODO: Jedním SQL dotazem zjistěte COUNT, AVG, MIN, MAX, SUM počtu obyvatel →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE mesta")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 5 — Kompletní úloha

Celý úkol je na vás od začátku do konce:

1. Připojte se k databázi.
2. Vytvořte tabulku `produkty` s atributy:
   - `id` — INT, PRIMARY KEY, AUTO_INCREMENT
   - `nazev` — VARCHAR(50), NOT NULL
   - `kategorie` — VARCHAR(30), NOT NULL
   - `cena` — DECIMAL(10, 2), NOT NULL
   - `pocet_skladem` — INT, NOT NULL, DEFAULT 0
3. Vložte **alespoň 5 produktů** s různými cenami a počty na skladě.
4. Vypište **všechny produkty** přehledně.
5. Vypište **průměrnou cenu**, **nejdražší produkt** (MAX ceny) a **celkový počet kusů na skladě** (SUM).
6. Smažte tabulku a odpojte se.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- Všechny produkty ---
1: Notebook (Elektronika) — 25999.99 Kč, skladem: 10
2: Myš (Elektronika) — 499.00 Kč, skladem: 50
...

--- Statistiky ---
Průměrná cena:        ...
Nejdražší produkt:    ...
Celkem kusů skladem:  ...
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# TODO: Celý úkol je na vás →


# Nezapomeňte na konci smazat tabulku a odpojit se!